# AQPG V17 Remediation — Phase 21A: Google Colab GPU Fine-Tuning

**Objective**: Clean GPU Fine-Tuning of `google/flan-t5-small` on NVIDIA GPU using verified 50,000-record V17 dataset.

> **SAFETY MANDATE**: All pre-flight checks must PASS before model training is launched. FastAPI integration remains **BLOCKED** (`approved_for_fastapi: false`).

In [ ]:
# Step 1: Install Dependencies & Mount Google Drive
!pip install -q transformers datasets accelerate torch

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Configure Paths & Variables
import os, sys, hashlib, json, torch, random
import numpy as np

DRIVE_ROOT = '/content/drive/MyDrive/AQPG'
TRAIN_DATASET = os.path.join(DRIVE_ROOT, 'phase21_step3_v17_train_dataset.jsonl')
VALIDATION_DATASET = os.path.join(DRIVE_ROOT, 'phase21_step3_v17_validation_dataset.jsonl')
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'backend/ml/models/checkpoints/flan_t5_v17_gpu')

EXPECTED_TRAIN_SHA = 'F150ED036D3B1612FB307949323972C40DD7074C8DC3C885621A72B993548C85'
EXPECTED_VAL_SHA = '7038D9C15A68269574E413A6C7E7E491A6DF047CD85F3BF9C3FFD82E7B764A3E'

print('Configured Drive Root:', DRIVE_ROOT)

In [ ]:
# Step 3: MANDATORY PRE-FLIGHT AUDIT CELL
def compute_sha256(filepath):
    with open(filepath, 'rb') as f: return hashlib.sha256(f.read()).hexdigest().upper()

print('='*75)
print('         AQPG V17 GOOGLE COLAB PRE-FLIGHT AUDIT')
print('='*75)
assert torch.cuda.is_available(), '[FAIL] CUDA is unavailable! Enable GPU in Colab (Runtime -> Change runtime type -> T4 GPU)'
print('[PASS] CUDA Available: True | Device:', torch.cuda.get_device_name(0))

assert os.path.exists(TRAIN_DATASET), f'[FAIL] Train dataset missing: {TRAIN_DATASET}'
assert os.path.exists(VALIDATION_DATASET), f'[FAIL] Validation dataset missing: {VALIDATION_DATASET}'

train_sha = compute_sha256(TRAIN_DATASET)
val_sha = compute_sha256(VALIDATION_DATASET)

with open(TRAIN_DATASET, 'r', encoding='utf-8') as f: train_cnt = sum(1 for l in f if l.strip())
with open(VALIDATION_DATASET, 'r', encoding='utf-8') as f: val_cnt = sum(1 for l in f if l.strip())

print(f'[PASS] Train Records: {train_cnt:,} | SHA: {train_sha[:16]}...')
print(f'[PASS] Val Records:   {val_cnt:,} | SHA: {val_sha[:16]}...')

assert train_cnt == 40000 and val_cnt == 10000, '[FAIL] Record count mismatch!'
print('[PASS] FastAPI Integration Status: BLOCKED (approved_for_fastapi: false)')
print('='*75)
print('ALL PRE-FLIGHT CHECKS PASSED! READY FOR CLEAN GPU TRAINING.')

In [ ]:
# Step 4: LAUNCH CLEAN V17 GPU TRAINING RUN
# Execute train_flan_t5_v17_gpu.py
!python {DRIVE_ROOT}/phase21_v17_colab/train_flan_t5_v17_gpu.py